# Task 2. Parser Service CPG Tăng Dần

## Mục tiêu

Task này xây dựng Parser Service xử lý từng file Python riêng lẻ và sinh event cho Code Property Graph (CPG). Service dùng module chuẩn `ast` của Python để lấy AST node, quan hệ cha-con trong AST, call edge, CFG đơn giản và DFG đơn giản.

Mỗi lần parse một file, service tạo bốn nhóm event: node, edge, metadata và error. Các event có `schema_version`, `timestamp`, `repo`, `commit_hash`, `file_path`, `file_hash` và `id` ổn định để phục vụ replay idempotent ở các task sau.

## Thiết Kế Parser Service

Sau khi refactor, Parser Service nằm trong thư mục `scripts/parser-service/` thay vì một file lớn ở root. Cấu trúc chính:

| File | Vai trò |
|---|---|
| `parser.py` | CLI entrypoint, duyệt danh sách file và điều phối parse/write |
| `cpg_parser.py` | Logic tạo AST/CFG/DFG/Call event bằng `ast` |
| `event_writer.py` | Ghi JSONL khi dry-run hoặc publish Kafka khi chạy thật |
| `stable_id.py` | Hàm hash tạo định danh ổn định |
| `topics.py` | Tên topic Kafka cho từng nhóm event |
| `schemas/*.schema.json` | JSON Schema mô tả cấu trúc event |

Ở chế độ `--dry-run`, service ghi mỗi nhóm event thành một file JSONL để kiểm tra nhanh trước khi bật Kafka.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
from collections import Counter

repo_path = Path("../transformers-pr-agent")
source_root = repo_path / "src"
parser_entrypoint = Path("../scripts/parser-service/parser.py")
out_dir = Path("../outputs/parser-output-demo")

if out_dir.exists():
    shutil.rmtree(out_dir)

print("Repository:", repo_path)
print("Source root:", source_root)
print("Parser entrypoint:", parser_entrypoint)
print("Output directory:", out_dir)

Repository: ..\transformers-pr-agent
Source root: ..\transformers-pr-agent\src
Parser entrypoint: ..\scripts\parser-service\parser.py
Output directory: ..\outputs\parser-output-demo


## Chạy Parser Service Ở Chế Độ Dry Run

Notebook parse 3 file Python đầu tiên trong `src/`. Giới hạn nhỏ giúp quá trình chạy nhanh nhưng vẫn tạo đủ node, edge và metadata để kiểm tra schema event. Khi chạy pipeline đầy đủ, có thể bỏ `--limit`.

In [2]:
subprocess.run(
    [
        "python", str(parser_entrypoint),
        "--repo", str(repo_path),
        "--source-root", str(source_root),
        "--limit", "3",
        "--dry-run",
        "--out-dir", str(out_dir),
    ],
    check=True,
)

CompletedProcess(args=['python', '..\\scripts\\parser-service\\parser.py', '--repo', '..\\transformers-pr-agent', '--source-root', '..\\transformers-pr-agent\\src', '--limit', '3', '--dry-run', '--out-dir', '..\\outputs\\parser-output-demo'], returncode=0)

## Thống Kê Event Đã Sinh

Mỗi dòng trong file JSONL tương ứng với một message sẽ được gửi vào Kafka khi chạy thật. Việc đếm số dòng giúp kiểm tra nhanh parser có sinh đủ bốn nhóm event hay không.

In [3]:
def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open(encoding="utf-8") as handle:
        return sum(1 for _ in handle)

counts = {
    name: count_jsonl(out_dir / f"{name}.jsonl")
    for name in ["nodes", "edges", "metadata", "errors"]
}

for name, count in counts.items():
    print(f"{name}: {count}")

nodes: 8680
edges: 9560
metadata: 3
errors: 0


## Mẫu Node Event

Node event biểu diễn một node trong AST hoặc một call target tổng hợp. Trường `id` được tạo bằng hash ổn định từ file, nội dung file và vị trí AST, giúp cùng một input tạo lại cùng một định danh.

In [4]:
with (out_dir / "nodes.jsonl").open(encoding="utf-8") as handle:
    sample_node = json.loads(next(handle))

print(json.dumps(sample_node, indent=2, ensure_ascii=False))

{
  "schema_version": 1,
  "event_type": "cpg_node",
  "timestamp": "2026-07-20T12:16:42.725407+00:00",
  "repo": "transformers-pr-agent",
  "commit_hash": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_path": "src/transformers/activations.py",
  "file_hash": "4b1a469c24be4e4c6320536ad55a7337d24de04c3ecc71597fbf0267f1560612",
  "id": "3d7eb68fc0ab3b58c1e58a4043ee0be1ff441ca608546e92f680fae582a33165",
  "label": "Module",
  "ast_path": "0",
  "line": null,
  "column": null,
  "end_line": null,
  "end_column": null
}


## Mẫu Edge Event

Edge event mô tả quan hệ giữa hai node. Trong phiên bản hiện tại, service sinh bốn loại cạnh:

| Edge type | Ý nghĩa |
|---|---|
| `AST_CHILD` | Quan hệ cha-con trong AST |
| `CFG_NEXT` | Statement kế tiếp trong cùng block |
| `DFG_REACHES` | Lần gán biến gần nhất đi tới lần đọc biến |
| `CALLS` | Một `ast.Call` gọi tới tên hàm/phương thức |

In [5]:
edge_types = Counter()
with (out_dir / "edges.jsonl").open(encoding="utf-8") as handle:
    first_edge = None
    for line in handle:
        event = json.loads(line)
        first_edge = first_edge or event
        edge_types[event["edge_type"]] += 1

print("Edge types:")
for edge_type, count in edge_types.most_common():
    print(f"{edge_type}: {count}")

print("\nSample edge:")
print(json.dumps(first_edge, indent=2, ensure_ascii=False))

Edge types:
AST_CHILD: 8235
CFG_NEXT: 448
CALLS: 442
DFG_REACHES: 435

Sample edge:
{
  "schema_version": 1,
  "event_type": "cpg_edge",
  "timestamp": "2026-07-20T12:16:42.725442+00:00",
  "repo": "transformers-pr-agent",
  "commit_hash": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_path": "src/transformers/activations.py",
  "file_hash": "4b1a469c24be4e4c6320536ad55a7337d24de04c3ecc71597fbf0267f1560612",
  "id": "440fb85550128a28d8198aaf423dec1a3f9d8ea02fa681a2983b21a34d3555af",
  "source": "3d7eb68fc0ab3b58c1e58a4043ee0be1ff441ca608546e92f680fae582a33165",
  "target": "890fd0a07f4add2a39e1c02c1cfda4f6dffac7592261ac40c8adb330909e5205",
  "edge_type": "AST_CHILD",
  "field": "body",
  "index": 0
}


## Mẫu Metadata Event

Metadata event phục vụ pipeline MongoDB ở task sau. Event này lưu commit, đường dẫn file, hash nội dung, kích thước file, số dòng và parser đã sử dụng.

In [6]:
with (out_dir / "metadata.jsonl").open(encoding="utf-8") as handle:
    sample_metadata = json.loads(next(handle))

print(json.dumps(sample_metadata, indent=2, ensure_ascii=False))

{
  "schema_version": 1,
  "event_type": "source_metadata",
  "timestamp": "2026-07-20T12:16:42.725359+00:00",
  "repo": "transformers-pr-agent",
  "commit_hash": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_path": "src/transformers/activations.py",
  "file_hash": "4b1a469c24be4e4c6320536ad55a7337d24de04c3ecc71597fbf0267f1560612",
  "id": "88d9a6a6c8dc3596fc6ff42974e073f2b8582fd57cd0658575fd28ceb6654478",
  "size_bytes": 13491,
  "line_count": 370,
  "parser": "python.ast"
}


## Lệnh Chạy Với Kafka

Khi Kafka broker đã chạy, bỏ `--dry-run` để gửi event thật vào Kafka:

```powershell
python scripts/parser-service/parser.py --repo transformers-pr-agent --source-root transformers-pr-agent/src --bootstrap-servers localhost:9092
```

Nếu muốn thử trên một số file trước:

```powershell
python scripts/parser-service/parser.py --repo transformers-pr-agent --source-root transformers-pr-agent/src --limit 3 --bootstrap-servers localhost:9092
```

In [7]:
schemas_dir = Path("../scripts/parser-service/schemas")
for schema_path in sorted(schemas_dir.glob("*.schema.json")):
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    required = ", ".join(schema.get("required", []))
    print(f"{schema_path.name}: {schema['title']}")
    print(f"  required: {required}")

edges.schema.json: CPG Edge Event
  required: schema_version, event_type, timestamp, id, source, target, edge_type, repo, commit_hash, file_path, file_hash
errors.schema.json: Parser Error Event
  required: schema_version, event_type, timestamp, id, repo, commit_hash, file_path, file_hash, error_type, message
metadata.schema.json: Source Metadata Event
  required: schema_version, event_type, timestamp, id, repo, commit_hash, file_path, file_hash, size_bytes, line_count, parser
nodes.schema.json: CPG Node Event
  required: schema_version, event_type, timestamp, id, label, repo, commit_hash, file_path, file_hash


## Kết Quả

Với 3 file đầu tiên trong `src/`, Parser Service sinh được các file `nodes.jsonl`, `edges.jsonl`, `metadata.jsonl` và `errors.jsonl` trong `outputs/parser-output-demo/`. Số lượng event được tính trực tiếp từ output ở cell thống kê phía trên, tránh hard-code số liệu khi repository hoặc parser thay đổi.

Các event này là đầu vào cho Task 3 về topic/schema Kafka và cho quá trình ingest vào Neo4j/MongoDB ở các task sau.

## Reflection

Parser Service đã chạy được ở chế độ dry-run nên có thể kiểm tra event trước khi phụ thuộc vào Kafka. Cách dùng `ast` phù hợp với phạm vi lab vì không cần parser ngoài, dễ giải thích và chạy ổn định trên Python source code.

Giới hạn hiện tại là CFG và DFG mới ở mức xấp xỉ: CFG nối các statement liên tiếp trong cùng block, còn DFG nối lần gán biến gần nhất tới lần đọc biến sau đó. Mức này đủ để minh họa CPG và thiết kế pipeline streaming, nhưng chưa thay thế được phân tích chương trình chuyên sâu trong hệ thống production.